# 04. Поиск по смыслу: выбор модели эмбеддингов

Контекст — `решения.md`: 3.5 (поиск по смыслу), 2.5, 1.5 (русские и казахские запросы), 3.7 (E5).

- **Запросы и разметка** — `ml/src/eval/search_queries.json` (78 запросов: en 44, ru 26, kk 8), метрики — `ml/src/eval/search.py`.
  Recall@10 — доля найденных подходящих книг из min(10, их число); MRR — 1 / место первой подходящей.
- **Кандидаты:** `multilingual-e5-small`, `multilingual-e5-base`, `paraphrase-multilingual-MiniLM-L12-v2` (`ml/src/content/embeddings.py`).
  Для сравнения — TF-IDF: поиск по словам, без перевода и без исправления опечаток.
- Поиск — по **всему каталогу** (408 тыс. книг), текст книги — поле `text` (название + категории + аннотация).
- **Дописать запросы:** добавить в конец `search_queries.json` и перезапустить ноутбук — векторы каталога берутся из кэша
  `data/processed/embeddings/`, заново считаются только запросы (секунды).

In [1]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent / "src"))
from content.embeddings import MODELS, catalog, catalog_info, encode, load_model
from content.recommend import tfidf_vectors
from eval.metrics import paired_bootstrap
from eval.search import evaluate_search, informative, load_queries, resolve, search_vectors, summary

REPORTS = Path.cwd().parent / "reports" / "metrics"
pd.options.display.float_format = "{:.3f}".format
pd.options.display.max_colwidth = 60

books = pd.read_parquet(Path.cwd().parents[1] / "data" / "processed" / "books_clean.parquet",
                        columns=["parent_asin", "title", "author", "text", "features", "categories", "n_ratings"])
ids = books["parent_asin"].to_numpy()
queries = load_queries()
qids = [q["id"] for q in queries]
texts = [q["query"] for q in queries]
rel_before = resolve(queries, books, pooled=False)   # разметка до прогона
rel = resolve(queries, books)                         # + доразметка по выдаче моделей (pooling)
allowed = informative(books)
print(f"Книг в каталоге: {len(books):,}; запросов: {len(queries)}; видеокарта: {torch.cuda.get_device_name(0)}")
print(f"Книг «из одного названия» (нет аннотации и подраздела категорий): {(~allowed).sum():,} ({(~allowed).mean():.1%})")
display(pd.DataFrame(queries).groupby(["lang", "kind"]).size().unstack(fill_value=0))

W1103 15:54:33.886000 9468 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Книг в каталоге: 407,927; запросов: 78; видеокарта: NVIDIA GeForce RTX 3050 6GB Laptop GPU
Книг «из одного названия» (нет аннотации и подраздела категорий): 2,372 (0.6%)


kind,author,title,topic,typo
lang,,,,
en,4,14,16,10
kk,0,2,6,0
ru,0,12,12,2


## 1. Поиск всеми моделями и скорость

In [2]:
K = 10
raw, results, speed = {}, {}, []      # raw — по всему каталогу, results — без книг «из одного названия»

for key in MODELS:
    vecs = catalog(key)                       # из кэша; при первом запуске — расчёт (минуты)
    info = catalog_info(key)
    load_model(key)
    encode(key, texts[:4], "query")           # прогрев
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    q = np.vstack([encode(key, [t], "query") for t in texts])   # по одному, как на сайте
    q_gpu = (time.perf_counter() - t0) / len(texts) * 1000
    torch.set_num_threads(4)              # при 8 потоках (= логических ядер) маленькие модели в разы медленнее
    cpu = SentenceTransformer(MODELS[key]["name"], device="cpu")
    cpu.max_seq_length = MODELS[key]["max_len"]
    pre = MODELS[key]["query"]
    cpu.encode([pre + texts[0]])
    t0 = time.perf_counter()
    for t in texts:
        cpu.encode([pre + t])
    q_cpu = (time.perf_counter() - t0) / len(texts) * 1000
    t0 = time.perf_counter()
    raw[key] = dict(zip(qids, search_vectors(q, vecs, ids, K)))
    s_ms = (time.perf_counter() - t0) / len(texts) * 1000
    results[key] = dict(zip(qids, search_vectors(q, vecs, ids, K, allowed=allowed)))
    speed.append({"модель": key, "размерность": info["dim"], "токенов текста книги": info["max_len"],
                  "каталог (видеокарта), мин": info["seconds"] / 60,
                  "запрос, видеокарта, мс": q_gpu, "запрос, процессор, мс": q_cpu, "перебор каталога, мс": s_ms})
    del cpu

t0 = time.perf_counter()
X, vec = tfidf_vectors(books["text"])
tfidf_fit = time.perf_counter() - t0
t0 = time.perf_counter()
raw["tfidf"] = results["tfidf"] = dict(zip(qids, search_vectors(vec.transform(texts), X, ids, K)))
speed.append({"модель": "tfidf", "размерность": X.shape[1], "каталог (видеокарта), мин": tfidf_fit / 60,
              "перебор каталога, мс": (time.perf_counter() - t0) / len(texts) * 1000})
speed = pd.DataFrame(speed)
speed.to_csv(REPORTS / "search_speed.csv", index=False, encoding="utf-8")
speed.style.format(precision=1, na_rep="").hide(axis="index")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

модель,размерность,токенов текста книги,"каталог (видеокарта), мин","запрос, видеокарта, мс","запрос, процессор, мс","перебор каталога, мс"
e5-small,384,256.0,11.1,9.0,8.3,5.0
e5-base,768,256.0,27.0,7.1,19.1,5.1
minilm,384,128.0,6.3,6.6,7.6,3.2
tfidf,123133,,0.6,,,6.7


**Скорость.** Все три модели для сайта достаточно быстрые: эмбеддинг запроса — 7–8 мс на видеокарте; на процессоре (4 потока)
e5-small и MiniLM — 7–10 мс, e5-base — ~19 мс. Важно для ML-сервиса: при 8 потоках (= числу логических ядер ноутбука) маленькие
модели замедлялись до ~105 мс из-за накладных расходов на потоки — число потоков нужно ограничивать. Каталог офлайн: MiniLM — 6 мин,
e5-small — 11 мин, e5-base — 27 мин. Полный перебор 408 тыс. векторов на видеокарте — 3–6 мс; на сайте поиск делает HNSW-индекс pgvector.

## 2. Качество: Recall@10 и MRR

In [3]:
ORDER = ["все", "en", "ru", "kk", "title", "author", "topic", "typo"]

def table(res_by_model, relevant):
    rows = []
    for name, res in res_by_model.items():
        s = summary(evaluate_search(res, queries, relevant)).reindex(ORDER)
        rows.append(s.assign(модель=name).rename_axis("срез").reset_index())
    return pd.concat(rows)

n = summary(evaluate_search(results["tfidf"], queries, rel))["запросов"].reindex(ORDER)
main = table(results, rel)
wide = main.pivot_table(index="срез", columns="модель", values=["Recall@10", "MRR"], sort=False).reindex(ORDER)
wide.insert(0, "запросов", n)
main.to_csv(REPORTS / "search_models.csv", index=False, encoding="utf-8")
print("Итог: без книг «из одного названия», после доразметки")
display(wide)

Итог: без книг «из одного названия», после доразметки


запросов Recall@10                           MRR                     
модель           e5-small e5-base minilm tfidf e5-small e5-base minilm tfidf
срез                                                                        
все          78     0.765   0.801  0.529 0.242    0.730   0.723  0.501 0.197
en           44     0.832   0.905  0.534 0.416    0.770   0.819  0.557 0.342
ru           26     0.796   0.715  0.542 0.023    0.788   0.671  0.418 0.013
kk            8     0.300   0.508  0.463 0.000    0.317   0.369  0.463 0.000
title        28     0.857   0.929  0.536 0.250    0.627   0.661  0.403 0.140
author        4     0.525   0.425  0.075 0.500    0.875   0.675  0.250 0.708
topic        34     0.724   0.728  0.647 0.262    0.849   0.809  0.717 0.244
typo         12     0.750   0.833  0.333 0.083    0.581   0.642  0.200 0.028

In [4]:
# Значимость: парный bootstrap по запросам (одни и те же 78 запросов)
per_query = {name: evaluate_search(res, queries, rel).set_index("id") for name, res in results.items()}
rows = []
for a, b in [("e5-small", "e5-base"), ("e5-small", "minilm"), ("e5-base", "minilm"), ("e5-small", "tfidf")]:
    for m in ["Recall@10", "MRR"]:
        r = paired_bootstrap(per_query[a][m], per_query[b][m])
        rows.append({"сравнение": f"{a} − {b}", "метрика": m, "разница": r["разница"],
                     "95% ДИ": f"{r['low']:+.3f} … {r['high']:+.3f}", "значимо": "да" if r["low"] > 0 or r["high"] < 0 else "нет"})
pd.DataFrame(rows).style.format({"разница": "{:+.3f}"}).hide(axis="index")

сравнение,метрика,разница,95% ДИ,значимо
e5-small − e5-base,Recall@10,-0.035,-0.101 … +0.032,нет
e5-small − e5-base,MRR,+0.006,-0.055 … +0.074,нет
e5-small − minilm,Recall@10,+0.236,+0.135 … +0.337,да
e5-small − minilm,MRR,+0.229,+0.124 … +0.331,да
e5-base − minilm,Recall@10,+0.271,+0.166 … +0.371,да
e5-base − minilm,MRR,+0.223,+0.115 … +0.328,да
e5-small − tfidf,Recall@10,+0.523,+0.427 … +0.612,да
e5-small − tfidf,MRR,+0.533,+0.433 … +0.630,да


In [5]:
pq_tab = pd.concat({name: d["Recall@10"] for name, d in per_query.items()}, axis=1)
pq_tab.insert(0, "запрос", texts)
pq_tab.insert(1, "подходящих", [len(rel[i]) for i in qids])
pq_tab.to_csv(REPORTS / "search_per_query.csv", encoding="utf-8")
print("Recall@10 по каждому запросу:")
display(pq_tab)

Recall@10 по каждому запросу:


,запрос,подходящих,e5-small,e5-base,minilm,tfidf
id,,,,,,
en-t01,Where the Crawdads Sing,1,1.000,1.000,0.000,1.000
en-t02,The Silent Patient,1,1.000,1.000,1.000,0.000
en-t03,Project Hail Mary,1,1.000,1.000,1.000,0.000
en-t04,Sapiens,1,1.000,1.000,1.000,1.000
en-t05,Educated,1,0.000,1.000,0.000,0.000
...,...,...,...,...,...,...
ty-08,thinking fast and slwo,1,1.000,1.000,0.000,0.000
ty-09,the silent pateint,1,1.000,1.000,0.000,0.000
ty-10,sapeins brief histroy of humankind,1,1.000,1.000,1.000,0.000


## 3. Книги «из одного названия»

У ~0,6% книг каталога в тексте только название: нет аннотации и нет подраздела категорий (бывает только верхний уровень
вроде «Books on CD», «Deals in Books»). Вектор такого короткого текста оказывается близко к любому короткому запросу,
особенно на другом языке, и такие книги забивают выдачу («книга про привычки» → *The Book of Shadows*, *Body Count*).
Поэтому в смысловой поиск их не пускаем; по названию их находит обычный поиск (гибрид, 3.5).

In [6]:
var = pd.concat([table(raw, rel).assign(вариант="весь каталог"), main.assign(вариант="без книг «из одного названия»")])
cmp = var[var["модель"] != "tfidf"].pivot_table(index=["модель", "вариант"], columns="срез", values="Recall@10", sort=False)
display(cmp[["все", "en", "ru", "kk", "topic"]])

срез                                     все    en    ru    kk  topic
модель   вариант                                                     
e5-small весь каталог                  0.686 0.809 0.662 0.087  0.626
         без книг «из одного названия» 0.765 0.832 0.796 0.300  0.724
e5-base  весь каталог                  0.744 0.884 0.696 0.125  0.653
         без книг «из одного названия» 0.801 0.905 0.715 0.508  0.728
minilm   весь каталог                  0.533 0.534 0.550 0.475  0.656
         без книг «из одного названия» 0.529 0.534 0.542 0.463  0.647

**Вывод.** Книги «из одного названия» сильнее всего мешают e5 на русских и казахских запросах: без них Recall@10 e5-small
по-русски 0,66 → 0,80, по-казахски 0,09 → 0,30; у e5-base по-казахски 0,13 → 0,51. У MiniLM (длина текста 128 токенов) эффекта нет.
Фильтр входит в метод: эти книги не попадают в смысловой индекс (и в `book_embeddings.parquet`), их находит обычный поиск по названию.

## 4. Доразметка по выдаче моделей (pooling)

Правила для тем и ручные списки находят не все подходящие книги, а правила опираются на английские слова —
это занижает смысловые модели. После прогона неразмеченные книги из топ-10 **каждой** модели по темам (все языки,
оба варианта каталога) просмотрены вручную, подходящие записаны в `pooled` в `search_queries.json` (169 книг).

In [7]:
before = table(results, rel_before).pivot_table(index="модель", columns="срез", values="Recall@10", sort=False)
after = main.pivot_table(index="модель", columns="срез", values="Recall@10", sort=False)
print("Recall@10 до и после доразметки:")
display(pd.concat({"до": before[["все", "en", "ru", "kk", "topic"]], "после": after[["все", "en", "ru", "kk", "topic"]]}, axis=1))
print(f"Подходящих книг по темам: до {sum(len(rel_before[i]) for i, q in zip(qids, queries) if q['kind'] == 'topic'):,}, "
      f"после {sum(len(rel[i]) for i, q in zip(qids, queries) if q['kind'] == 'topic'):,}")

Recall@10 до и после доразметки:


до                         после                        
срез       все    en    ru    kk topic   все    en    ru    kk topic
модель                                                              
e5-small 0.688 0.761 0.696 0.263 0.547 0.765 0.832 0.796 0.300 0.724
e5-base  0.718 0.827 0.623 0.421 0.537 0.801 0.905 0.715 0.508 0.728
minilm   0.454 0.475 0.427 0.425 0.474 0.529 0.534 0.542 0.463 0.647
tfidf    0.213 0.366 0.019 0.000 0.194 0.242 0.416 0.023 0.000 0.262

Подходящих книг по темам: до 26,888, после 27,127


## 5. Выбор модели

**Рекомендация — `multilingual-e5-base`** (окончательный выбор — за автором, см. решения.md 3.5).
- **По качеству поиска e5-base и e5-small статистически неразличимы:** Recall@10 0,80 и 0,77, MRR 0,72 и 0,73; интервал разницы
  по парному bootstrap по запросам содержит 0 (Recall −0,10 … +0,03). Но e5-base впереди почти во всех срезах: английский (0,91 / 0,83),
  точные названия (0,93 / 0,86), опечатки (0,83 / 0,75), казахский (0,51 / 0,30, всего 8 запросов); e5-small лучше только по-русски
  (0,80 / 0,72).
- **В content-рекомендациях (05) e5-base значимо лучше** e5-small — на 24%. Одна модель на поиск и «похожие» проще в поддержке,
  поэтому решает сумма.
- **Цена e5-base приемлема:** запрос на процессоре ~19 мс вместо ~10; векторы 768 вместо 384 — вдвое больше места в pgvector
  (~1,2 ГБ векторов на 405 тыс. книг против ~0,6 ГБ) и индекс HNSW; каталог считается 27 мин вместо 11, но это офлайн и редко.
- **MiniLM значимо хуже** (Recall@10 0,53): плохо находит книги по названию (0,54) и опечатки (0,33). Наравне с e5 только по-казахски.
- **TF-IDF** (0,24) находит только английские запросы со словами из текста книги: русские и казахские — ноль, опечатки — почти ноль.
  Он нужен как часть гибрида для точных совпадений, а не как замена смысловому поиску.
- **Слабое место — казахский:** 0,30–0,51 на 8 запросах; переведённые названия находятся плохо. Записано как ограничение (1.5);
  проверим ещё раз, когда автор добавит казахские запросы.
- **Доразметка (pooling)** подняла Recall@10 у всех моделей (e5-small 0,69 → 0,77, e5-base 0,72 → 0,80, TF-IDF 0,21 → 0,24) и
  не изменила порядок моделей.
- **Ограничения разметки:** 10–20% шума в правилах для тем (категории Amazon неточные); у *Atomic Habits* в каталоге нет аннотации —
  находится по названию, но не по смыслу («книга про привычки»).

## 6. Примеры выдачи

In [8]:
title = dict(zip(books["parent_asin"], books["title"]))
for qid in ["ru-t01", "ru-d09", "kk-01", "kk-05", "en-d02", "ty-02", "en-t14"]:
    q = next(x for x in queries if x["id"] == qid)
    relevant = set().union(*rel[qid])
    print(f"\n«{q['query']}»")
    for name in ["e5-small", "e5-base", "minilm", "tfidf"]:
        top = [("+ " if i in relevant else "  ") + str(title[i])[:40] for i in results[name][qid][:3]]
        print(f"  {name:8} " + " | ".join(top))


«грокаем алгоритмы»
  e5-small + Grokking Algorithms: An Illustrated Guid |   Algorithms Illuminated (Part 2): Graph A |   Algorithms in a Nutshell (In a Nutshell 
  e5-base  + Grokking Algorithms: An Illustrated Guid |   R Graphs Cookbook |   Genetic Algorithms in Search, Optimizati
  minilm     Algorithms Illuminated (Part 2): Graph A |   Design and Analysis of Computer Algorith |   Algorithms (4th Edition)
  tfidf    

«книга про привычки»
  e5-small + The Power of Positive Habits |   Voices of Recovery: A Daily Reader |   This Book is About You: Discover, Decode
  e5-base  + Elizabeth Gaskell : A Habit of Stories | + Learning and Leading with Habits of Mind | + Habits: The Mother's Secret to Success (
  minilm   + The Power of Habit: Why We Do What We Do | + The 8th Habit: From Effectiveness to Gre |   The Creature of Habit
  tfidf    

«ғарыш туралы кітаптар»
  e5-small   The Aerodynamics of Biscuits |   Book of the Eskimos | + Cosmos: A Field Guide
  e5-base    Ripple Effect |  

Примеры показывают типичные случаи: перевод названия («грокаем алгоритмы» → *Grokking Algorithms*) e5 решают сразу, TF-IDF — нет.
По-казахски темы находятся частично («ғарыш туралы кітаптар» — плохо), а переведённое название «Гарри Поттер және философиялық тас»
находят только e5-base и MiniLM — британское издание *Harry Potter and the Philosopher's Stone*. Для казахских книг поэтому важен
свой каталог с названиями в оригинале (1.5).